Question 1: Install Spark and PySpark

In [1]:
# 1) Parar la sesión actual si existe
try:
    spark.stop()
except NameError:
    pass
# 2) Forzar Java 17 en este kernel
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# 3) Verifica que ahora ve Java 17
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)


In [2]:
import pyspark
from pyspark.sql import SparkSession


In [3]:

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 03:06:55 WARN Utils: Your hostname, codespaces-29ba17, resolves to a loopback address: 127.0.0.1; using 10.0.1.46 instead (on interface eth0)
26/03/08 03:06:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 03:06:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:

print(f"Spark version: {spark.version}")


Spark version: 4.1.1


Question 2: Yellow November 2025

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 02:57:23--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.228, 3.167.84.131, 3.167.84.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   144MB/s    in 0.5s    

2026-03-08 02:57:24 (144 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [5]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")


In [6]:
df = df.repartition(4)

In [7]:
df.write.parquet('yellow/2025/11/')

In [22]:
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

Question 3: Count records

In [8]:
from pyspark.sql import functions as F

df = spark.read.parquet("yellow/2025/11/")

df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15")) \
  .select("tpep_pickup_datetime") \
  .count()

162604

Question 4: Longest trip

In [10]:
df.withColumn(
    "trip_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600.0
).agg(F.max("trip_hours").alias("max_hours")).show()

+-----------------+
|        max_hours|
+-----------------+
|90.64666666666666|
+-----------------+



Question 5: User Interface
Respsonse: 4040

Question 6: Least frequent pickup location zone

In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 03:20:45--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.131, 3.167.84.86, 3.167.84.228, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.131|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 03:20:45 (1.19 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [12]:
lookup_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("taxi_zone_lookup.csv"))

lookup_df.createOrReplaceTempView("zones")

# Ejemplo rápido para verificar
spark.sql("SELECT * FROM zones LIMIT 5").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [24]:
least_pickup_zone = (
    df.join(lookup_df, df.PULocationID == lookup_df.LocationID, "inner")
      .groupBy("Zone")
      .count()
      .orderBy(F.col("count").asc())
      .limit(10)
)

least_pickup_zone.show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Eltingville/Annadale/Prince's Bay            |1    |
|Governor's Island/Ellis Island/Liberty Island|1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+



In [25]:
spark.stop()